In [ ]:
%load_ext autoreload
%autoreload 2

After testing the effectiveness of using the MODIS product to classify fires from the GFA dataset, this notebook uses MODIS to classify fires in the MTBS and GFD datasets, which do not originally contain a land cover attribute.

In [ ]:
import geopandas as gpd

from wildfire_stats import config as cfg
from wildfire_stats.data import classification as dcls
from wildfire_stats.data import io as dio
from wildfire_stats.data import rasters as drast
from wildfire_stats.fitting import fits as fits
from wildfire_stats.plotting import distributions as pdist
from wildfire_stats.reporting import tables as rtab

P = cfg.load_paths()
A = cfg.load_assumptions()

modis_by_year = drast.index_modis_by_year(P.modis / "GeoTIFF")

### Classify MTBS

In [ ]:
mtbs_gdf = dio.load_mtbs(P.mtbs / "S_USA.MTBS_BURN_AREA_BOUNDARY.shp")

mtbs_filtered = mtbs_gdf[mtbs_gdf["AREA_SQKM"] >= A["min_fire"]].copy()
mtbs_filtered = mtbs_filtered.rename(columns={
    "YEAR": "year",
    "AREA_SQKM": "area_km2"
})

mtbs_classified = dcls.classify_with_modis(mtbs_filtered, "year", modis_by_year, out_col="modis_class")

print(f"Classified {len(mtbs_classified)} MTBS fires over {A['min_fire']} km²")
mtbs_classified[["year", "area_km2", "modis_class"]].head()

### Classify Idaho GFD

In [ ]:
idaho_gdf = dio.load_idaho(P.idaho / "HFD_HistoricFiresDatabase7_0.gdb")

idaho_filtered = idaho_gdf[idaho_gdf["area_km2"] >= A["min_fire"]].copy()
idaho_filtered = idaho_filtered.rename(columns={"FIRE_YEAR": "year"})

idaho_classified = dcls.classify_with_modis(idaho_filtered, "year", modis_by_year, out_col="modis_class")

These take a long time to run, so saving them below so you can call them in later.

In [ ]:
dio.save_vector(mtbs_classified, P.logged_output / "mtbs_2003_2016_classified_timevarying.shp")
dio.save_vector(idaho_classified, P.logged_output / "idaho_2003_2016_classified_timevarying.shp")

# written by notebook 3
gfa_classified = gpd.read_file(P.logged_output / "gfa_2003_2016_usa_classified_timevarying.shp")

Now, the following code runs the distribution fitting routine on each land cover type for each dataset.

In [ ]:
years = A["modis_years"]

# "modis_clas": the shapefile round-trip truncates GFA's modis_class column name
results_gfa = fits.fit_static_by_group(gfa_classified, "modis_clas", years=years)
results_idaho = fits.fit_static_by_group(idaho_classified, "modis_class", years=years)
results_mtbs = fits.fit_static_by_group(mtbs_classified, "modis_class", years=years)

dio.save_pickle(results_gfa, P.output / "gfa_static_fits_by_biome_2003_2016.pkl")
dio.save_pickle(results_mtbs, P.output / "mtbs_static_fits_by_biome_2003_2016.pkl")
dio.save_pickle(results_idaho, P.output / "idaho_static_fits_by_biome_2003_2016.pkl")

The following table summarizes the "good" fits for each dataset+land cover type. Good fits are ones fit on >400 fires, loglikelihood relative to the best fit is <=2.0, and they do not reduce to a simpler distribution with 95% confidence. This includes an additional filter to remove fits where the relative error on a parameter was too big, but that max_rel_error constraint was not used in future analysis.

In [ ]:
results_dicts = {
    "GFA": results_gfa,
    "Idaho": results_idaho,
    "MTBS": results_mtbs
}

counts_dicts = {
    "GFA": gfa_classified[(gfa_classified["year"].between(*years)) & (gfa_classified["area_km2"] >= A["min_fire"])]["modis_clas"].value_counts(),
    "Idaho": idaho_classified[(idaho_classified["year"].between(*years)) & (idaho_classified["area_km2"] >= A["min_fire"])]["modis_class"].value_counts(),
    "MTBS": mtbs_classified[(mtbs_classified["year"].between(*years)) & (mtbs_classified["area_km2"] >= A["min_fire"])]["modis_class"].value_counts(),
}

rtab.build_comparison_table(results_dicts, counts_dicts)

The following function visualizes the "good" fits in parameter space for a all datasets and land cover types.

In [ ]:
pdist.plot_distribution_params(results_dicts, counts_dicts, log_axes=False)

In [ ]:
pdist.plot_modis_category_ccdf(mtbs_classified, "Savannas", which=["truncated_power_law", "power_law"])